In [1]:
# Initializes the configuration settings for the application
from config import settings

In [2]:
# Import necessary modules for the application
import sqlite3
import pandas as pd
from langchain_core.messages.ai import AIMessage
from langchain_community.utilities import SQLDatabase
from langchain.chat_models import init_chat_model
from langchain_core.prompts.prompt import PromptTemplate

/var/folders/wk/sr9xqbm946l7mm1086p_d22h0000gn/T/ipykernel_74815/2721866931.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
# Load the data schema with default 3 records
db_path = "./data/chinook.db"
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
schema_info = db.get_table_info()
print(schema_info)  


CREATE TABLE albums (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES artists ("ArtistId")
)

/*
3 rows from albums table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE artists (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from artists table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE customers (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES employe

In [4]:
# Init our llm model
chat_model = init_chat_model(f"openai:{settings.openai_llm_model}")

In [5]:
# Sql Query generation template for our chat model
sql_query_generation_prompt = PromptTemplate(
    template="""You are a helpful sql query generator for a SQL database, based on user query. 
    You will be provided with the following information about database. 
    Your goal is to generate require sql statements.
    ### Database Schema:
    {schema_info} 

    ### User Query:
    {user_query}

    ### Requirements:
    - Output only the SQL query, no explanations or additional text
    - Use proper SQLite syntax and features only
    - Query for at most top k results using LIMIT clause unless a specific number is requested
    - Order results to return the most informative data
    - Only query columns that are needed to answer the question
    - Wrap column names in double quotes (") as delimited identifiers
    - Use only column names that exist in the schema below
    - Pay attention to which column is in which table
    - Use date('now') for current date when queries involve "today"

    ### Query Construction Guidelines:
    - Use appropriate joins between tables when needed
    - Include all relevant filtering conditions
    - Handle complex requirements including sorting, filtering, and aggregation
    - Use subqueries or CTEs (Common Table Expressions) when appropriate
    - Ensure queries are optimized for performance
    """,
    input_variables=["user_query"],
    partial_variables={"schema_info": schema_info},
)


In [6]:
# Testing
user_query = "List me all the available genres, give me any 4"
generate_sql_query_pipeline = sql_query_generation_prompt | chat_model
response = generate_sql_query_pipeline.invoke({"user_query": user_query})
print(response.text)

```sql
SELECT "GenreId", "Name" 
FROM genres 
LIMIT 4;
```


In [7]:
# Query builder pipeline
generate_sql_query_pipeline = sql_query_generation_prompt | chat_model
response = generate_sql_query_pipeline.invoke({"user_query": user_query})
raw_sql_query = response.text
print(raw_sql_query)

```sql
SELECT "GenreId", "Name" 
FROM genres 
LIMIT 4;
```


In [8]:
# Cleaning and validating the generated SQL query
def clean_sql_query(raw_sql_query: AIMessage) -> str:
    """Clean up the SQL query by removing unnecessary characters and formatting."""
    raw_sql_query_text = raw_sql_query.text.strip()
    if raw_sql_query_text:
        cleaned_sql = raw_sql_query_text.replace("```sql", "").replace("```","").strip()
        print(f"Cleaned SQL Query: {cleaned_sql}")
        return cleaned_sql
    return raw_sql_query_text

def validate_sql_query(sql_query:str)->str:
    """Validate the generated SQL query to ensure it's safe and doesn't contain dangerous keywords."""
    BLACKLISTED_KEYWORDS = ['DELETE', 'DROP', 'TRUNCATE', 'INSERT', 'UPDATE', 'CREATE', 'ALTER']
    if any(keyword in sql_query.upper() for keyword in BLACKLISTED_KEYWORDS):
        raise ValueError(f"Invalid SQL query: {sql_query}")
    return sql_query

In [9]:
# Generate a final query before retrieving
generate_final_query_pipeline = generate_sql_query_pipeline | clean_sql_query | validate_sql_query
cleaned_sql_text = generate_final_query_pipeline.invoke({"user_query": user_query})
print(cleaned_sql_text)

Cleaned SQL Query: SELECT "GenreId", "Name" FROM genres LIMIT 4;
SELECT "GenreId", "Name" FROM genres LIMIT 4;


In [10]:
# Retrieve the data from the database using the cleaned SQL text
def retrieve_data_from_database(sql_query:str)->pd.DataFrame:
    """Retrieve data from the database based on the provided SQL query."""
    try:
        print("Executing SQL query to retrieve data from the database...")
        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query(sql_query, conn)
            return df
    except Exception as e:
        print(f"Error retrieving data: {e}")
        return pd.DataFrame()

In [11]:
# Final retrieval pipeline
retrieve_data_pipeline = generate_final_query_pipeline | retrieve_data_from_database
df = retrieve_data_pipeline.invoke({"user_query": user_query})
df.head(5)

Cleaned SQL Query: SELECT "GenreId", "Name" FROM genres LIMIT 4;
Executing SQL query to retrieve data from the database...


,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk


In [12]:
# Augment prerequisites with the retrieved data
def format_df_to_str(dataframe:pd.DataFrame)->str:
    """Format a pandas DataFrame into a string representation."""
    if dataframe.empty:
        return "No result found."
    return dataframe.to_string(index=False)

In [13]:
# Final response system prompt
final_response_system_promt = PromptTemplate(template="""
You are an AI assistant that helps users with their questions about structured data. 
You have access to a database containing structured data.

###Data:
{data}
Your task is to create a friendly, informative response to the user's question based on the provided data.
Use the below points, while creating a response:
1. Highlight key information in bold.
2. End with an offer to provide more information if needed.
3. Present data in a more conversational way.
4. If there are no results, say so and ask for clarification.

###User Query: 
{user_query}
""",
input_variables=["data", "user_query"])

In [14]:
# Final pipeline (Retrieval + Augement + Generate) Test
data = (generate_final_query_pipeline | retrieve_data_from_database | format_df_to_str).invoke({"user_query": user_query})
response = (final_response_system_promt | chat_model).invoke({"data": data, "user_query": user_query})
print(response.text)

Cleaned SQL Query: SELECT "GenreId", "Name" FROM genres ORDER BY "GenreId" LIMIT 4;
Executing SQL query to retrieve data from the database...
Absolutely! Here are some of the available music genres from our collection:

1. **Rock**
2. **Jazz**
3. **Metal**
4. **Alternative & Punk**

If you’d like to know more about any specific genre or need additional information, just let me know!


In [15]:
def generate_response(user_query: str):
    """ Generate a response to the user's query based on structured data. """
    data = (generate_final_query_pipeline | retrieve_data_from_database | format_df_to_str).invoke({"user_query": user_query})
    response = (final_response_system_promt | chat_model).invoke({"data": data, "user_query": user_query})
    return response.text

In [16]:
user_question = "List all the customers from brazil"
print(generate_response(user_query=user_question))

Cleaned SQL Query: SELECT "CustomerId", "FirstName", "LastName", "Email" 
FROM customers 
WHERE "Country" = 'Brazil'
ORDER BY "LastName", "FirstName";
Executing SQL query to retrieve data from the database...
Here are the customers from Brazil that I found in our records:

1. **Roberto Almeida**  
   - **Email:** roberto.almeida@riotur.gov.br  

2. **Luís Gonçalves**  
   - **Email:** luisg@embraer.com.br  

3. **Eduardo Martins**  
   - **Email:** eduardo@woodstock.com.br  

4. **Fernanda Ramos**  
   - **Email:** fernadaramos4@uol.com.br  

5. **Alexandre Rocha**  
   - **Email:** alero@uol.com.br  

If you need more details or have any other questions, feel free to ask!
